# Complaint Explorer Modeling (Negative Reviews Only)

Phase 1: BERTopic modeling on negative reviews only, topic reduction to dashboard size, and NPMI evaluation.

In [1]:
import os, sys
if '..' not in sys.path:
    sys.path.insert(0, '..')
import numpy as np, pandas as pd
from bertopic import BERTopic
from src.topics import count_topics, build_gensim_dictionary
from src.complaints import (
    load_negative_reviews,
    create_complaints_bertopic,
    fit_complaints_bertopic,
    save_complaints_model,
    reduce_complaints_topics,
    compute_complaint_coherence,
    sample_theme_reviews,
)

In [2]:
# Load negative reviews and verify 1-to-1 embedding alignment
neg_df, neg_emb, is_aligned = load_negative_reviews(
    clean_path='../data/processed/reviews_clean.parquet',
    embeddings_path='../data/processed/embeddings.npy',
)
print('Negative reviews count:', len(neg_df))
print('Alignment verified:', is_aligned)
docs = neg_df['text'].tolist()

In [3]:
# Fit BERTopic model on negative reviews
model = create_complaints_bertopic(random_state=42)
model, topics, probs, fit_duration = fit_complaints_bertopic(model, docs, neg_emb)
print(f'Fit completed in: {fit_duration:.2f} seconds')
print('Unmerged Topic Count (excluding -1):', count_topics(model))
print('Unmerged Outlier Fraction:', (np.array(topics) == -1).mean())

# Save unmerged model and assignments
save_complaints_model(
    topic_model=model,
    model_dir='../models/bertopic_neg',
    df=neg_df,
    topics=topics,
    parquet_path='../data/processed/topics_neg.parquet',
    col_name='topic_neg',
)

In [4]:
# Reduce to at most 15 themes on fresh load (Rule 5)
fresh_model = BERTopic.load('../models/bertopic_neg')
merged_model, merged_topics = reduce_complaints_topics(fresh_model, docs, nr_topics=15)
print('Merged Topic Count (excluding -1):', count_topics(merged_model))
print('Merged Outlier Fraction:', (np.array(merged_topics) == -1).mean())

# Save merged model and assignments
save_complaints_model(
    topic_model=merged_model,
    model_dir='../models/bertopic_neg_merged',
    df=neg_df,
    topics=merged_topics,
    parquet_path='../data/processed/topics_neg_merged.parquet',
    col_name='topic_neg_merged',
)

In [5]:
# Compute NPMI (c_npmi) coherence on negative reviews corpus
tokens_df = pd.read_parquet('../data/processed/tokens.parquet')
neg_tokens_df = tokens_df[tokens_df['review_id'].isin(neg_df['review_id'])].sort_values('review_id')
neg_texts = neg_tokens_df['tokens'].tolist()
dictionary = build_gensim_dictionary(neg_texts, no_below=10, no_above=0.4)

s_unm, n_unm, o_unm = compute_complaint_coherence(
    model_or_path='../models/bertopic_neg',
    docs=docs,
    topics=topics,
    texts=neg_texts,
    dictionary=dictionary,
    processes=1,
)
s_mrg, n_mrg, o_mrg = compute_complaint_coherence(
    model_or_path='../models/bertopic_neg_merged',
    docs=docs,
    topics=merged_topics,
    texts=neg_texts,
    dictionary=dictionary,
    processes=1,
)
summary_df = pd.DataFrame([
    {
        'model': 'BERTopic negative (unmerged)',
        'topics': n_unm,
        'outlier_fraction': f'{o_unm * 100:.2f}%',
        'npmi': round(s_unm, 4),
        'note': 'These scores come from a different corpus (negative reviews only) and are NOT directly comparable with the earlier table.',
    },
    {
        'model': f'BERTopic negative (merged {n_mrg} themes)',
        'topics': n_mrg,
        'outlier_fraction': f'{o_mrg * 100:.2f}%',
        'npmi': round(s_mrg, 4),
        'note': 'These scores come from a different corpus (negative reviews only) and are NOT directly comparable with the earlier table.',
    },
])
summary_df.to_csv('../results/complaint_topic_summary.csv', index=False)
summary_df

In [6]:
# Inspect merged themes with 8 sample reviews each
merged_fresh = BERTopic.load('../models/bertopic_neg_merged')
merged_info = merged_fresh.get_topic_info()
valid_themes = merged_info[merged_info['Topic'] != -1].sort_values('Count', ascending=False)

for _, row in valid_themes.iterrows():
    tid = int(row['Topic'])
    cnt = int(row['Count'])
    rep_words = [w for w, _ in merged_fresh.get_topic(tid)[:10]]
    print(f'\nTHEME ID: {tid} | COUNT: {cnt} | SHARE: {cnt / len(neg_df) * 100:.2f}%')
    print(f'TOP 10 KEYWORDS: {", ".join(rep_words)}')
    print('-' * 70)
    sample_df = sample_theme_reviews(neg_df, merged_topics, topic_id=tid, n=8, random_state=1, char_limit=150)
    for s_idx, (_, s_row) in enumerate(sample_df.iterrows(), 1):
        print(f'  [{s_idx}] Review {s_row["review_id"]}: {s_row["text_150"].replace(chr(10), " ").strip()}')